# When should an always-on agent wake up?

**Predict:** a constructed score of 0.08 puts a worrying event below the tuning-selected quiet threshold 0.20. Would you defer it?

All events, labels, scores and prices are teaching inputs. This notebook executes a routing policy and completed-decision journal; it does not measure a model or deployed agent. No network, credentials, model download, GPU or extra packages.

Keep `wake_gate_v3.py` and `synthetic-events-v1.json` beside this notebook. Python 3.10 or newer. The loader checks both source hashes before execution.


In [ ]:
from pathlib import Path
import os, json, hashlib, importlib.util, uuid
HERE = Path.cwd()
HELPER_SHA256 = '13556c06a245ac9bab2ac94800113d5d2c929ae534c853c2883d1a878be4c967'
FIXTURE_SHA256 = 'f9f3bad994cde2e946ee43330a84c865ccba17f6718e7a680d8a8d6edb82083a'
for name, expected in [("wake_gate_v3.py", HELPER_SHA256), ("synthetic-events-v1.json", FIXTURE_SHA256)]:
    assert hashlib.sha256((HERE / name).read_bytes()).hexdigest() == expected, f"Changed teaching artifact: {name}"
spec = importlib.util.spec_from_file_location("profrod_wake_gate", HERE / "wake_gate_v3.py")
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
RUN_ROOT = Path(os.environ.get("PROFROD_LAB_OUTPUT", "./lab-runs"))
RUN_DIR = RUN_ROOT / ("notebook-" + uuid.uuid4().hex)
report = lab.run_lab(RUN_DIR)
print(json.dumps({"threshold": report["threshold_frozen_from_tune"], "model_calls": report["model_calls"], "test": {k:v for k,v in report["results"]["test"].items() if k != "decisions"}}, indent=2))


## The tuning pass did not certify the test set

Sixteen events select the quiet threshold; 24 distinct episodes test it. The selected 0.20 has zero important misses on tuning, but 1 of 10 known-important test episodes is deferred. This is an intentionally constructed counterexample, not an estimate of a model's deployment miss rate.

Inspect the event rather than editing its output to make the conclusion look better.

The upper investigation threshold 0.8 is fixed for the exercise; only the lower quiet threshold is selected on tuning cases. Neither is a general operating recommendation.


In [ ]:
events = json.loads((HERE / "synthetic-events-v1.json").read_text())["events"]
test = [e for e in events if e["split"] == "test"]
trap = next(e for e in test if e["event_id"] == "test-important-01")
print(json.dumps(trap, indent=2))
print("Frozen route:", lab.route(trap, report["threshold_frozen_from_tune"]))
print("Intervention route at0.05:", lab.route(trap, 0.05))
assert lab.route(trap, .20)[0] == "defer"
assert lab.route(trap, .05)[0] == "review"


## Change the threshold and inspect the trade

The sweep below is an explanatory intervention using the test data. Once you use it to choose a new threshold, these episodes cannot provide untouched test evidence for that revised policy. Reserve another episode set before claiming improvement.

The positive denominator includes protected/bypassed and failed-response cases. Brier uses only 18 valid attempted binary judgments; ambiguous labels, bypasses, and failures remain in route/cost reports.


In [ ]:
for threshold in [0.02, 0.05, 0.10, 0.20, 0.30]:
    r = lab.evaluate(test, threshold)
    print({"threshold": threshold, "routes": r["routes"], "misses": r["missed_important_events"], "important": r["important_events"], "synthetic_USD_cents": r["synthetic_cascade_cost_cents"]})


## A cheap call can create an expensive cascade

All unit costs are **synthetic USD integer cents**, not Jev prices or invoices: 1 cent/gate attempt, 25 cents/investigation, 50 cents/review. Trusted bypasses avoid the gate; failed attempts count. No real investigator or human review result is measured.

Expected result: 22×1+2×25+12×50=672 cents versus 24×25=600 cents. Actual model/API charges for this offline run: USD 0.00. The gate misses important work too, so the routes do not have matched quality.


In [ ]:
r = report["results"]["test"]
assert r["synthetic_cascade_cost_cents"] == 22*1 + 2*25 + 12*50 == 672
assert r["synthetic_investigate_every_event_cost_cents"] == 24*25 == 600
assert r["valid_binary_judgments"] == 18
print("Synthetic cascade USD:", r["synthetic_cascade_cost_cents"] / 100)
print("Synthetic all-investigation USD:", r["synthetic_investigate_every_event_cost_cents"] / 100)
print("Controls:", json.dumps(report["controls"], indent=2))


## Inspect a Jev request without sending it

The official interface and pinned model identifier are cited in the article and source ledger. The same event/label/split protocol is the basis for a later comparison against NLI, GLiClass, Laya, and Jev. Their score semantics must be checked separately. The notebook does not provide or exercise live authentication, billing, or an API round trip.

No fixture score, expected label, or trusted approval control belongs in the model state. Missing/malformed responses must route to review. Choice confidence is a distribution statistic; it is not a measured correctness probability. Noul has a yes probability and no separate confidence field.


In [ ]:
payload = lab.request_payload(trap)
print(json.dumps(payload, indent=2))
assert set(payload["state"]) == {"source", "text"}
constructed_response = {"model": lab.MODEL_ID, "answers": {"needs_attention": {"type": "noul", "noul": .08}}}
print("Constructed response parse:", lab.parse_jev_response(constructed_response))


## Check the invariant, not the product name

A confidence score cannot grant permission. The trusted-control case asks for review even when untrusted text requests a different action. SQLite verifies occurrence identity and restart/replay behavior; this is not a production scheduler, lease, fenced executor, or durable-storage certification. Deferral in a real controller needs an explicit review deadline and ungated audit.

Next: [durable inbox](https://www.profrod.ai/book/ch08-durable-work), [isolation](https://www.profrod.ai/book/ch15-isolation), and [evaluation](https://www.profrod.ai/book/ch16-evaluation).

Question for review: what does the output prove, and which claim would still require a real model experiment?

The offline replay scores first and journals completed decisions afterward. The restart check covers that journal, not durable admission before scoring or interruption during judgment.
